In [1]:
import os 
import sys
import json
import pickle
from pathlib import Path

from tqdm import tqdm 
import numpy as np 
import torch
import torch.nn.functional as f
from torch.utils.data import dataset, dataloader
from transformers import AutoTokenizer, AutoModel
import datasets
from datasets import load_dataset

import data_utils

In [ ]:
dataSet = 'validation' # validation dev test
dir_name = dataSet.strip('idation')

data_path = Path(f'../data/mmmu/{dir_name}')
if data_path not in sys.path:
    sys.path.append(data_path)

os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + f":{dir_name}"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# load mmmu dataset
dataset = load_dataset("lmms-lab/MMMU", split=f"{dataSet}")

# load saved conversation with gpt and attach to dataset
gpt_conversation_path = data_path / f'mmmu_{dir_name}_gpt4o_response_v2.jsonl' #TODO: actually the test set
gpt_conversations = []
with open(gpt_conversation_path, 'r') as f:
    for line in f:
        # each line is a json
        gpt_conversations.append(line.strip().strip('"'))
data_conversation = datasets.Dataset.from_dict({"conversations": gpt_conversations})
dataset = datasets.concatenate_datasets([dataset, data_conversation], axis=1)

# # load selected hashtags
# selected_hashtags_path = data_path / 'embeddings/single_keyword_embeddings_dict_gte-base-en-v1.5.pkl'
# with open(selected_hashtags_path, 'rb') as f:
#     selected_hashtags = pickle.load(f)
# print(len(selected_hashtags[0]))
# print(len(selected_hashtags[1]))
# print(len(selected_hashtags[2]))

# load keywords
keyword_dir = Path(f'../keyword/mmmu_gpt/{dir_name}_keyword')
keyword_test_list = data_utils.get_extracted_keywords(keyword_dir)
data_keyword = datasets.Dataset.from_dict({"keywords": keyword_test_list})
print(len(data_keyword))
print(len(dataset))
# test_dataset = datasets.concatenate_datasets([test_dataset, data_keyword], axis=1)
dataset = datasets.concatenate_datasets([dataset, data_keyword], axis=1)
# only keep single image questions
dataset_single_image = dataset.filter(lambda x: x['image_2'] is None)
print(len(dataset_single_image))

In [ ]:
dataset[149]

In [ ]:
import numpy as np
import torch

print("Torch version:", torch.__version__)


import clip
clip.available_models()

import os
from PIL import Image
import numpy as np
import torch

from collections import defaultdict
import numpy as np
import pickle
from tqdm import tqdm
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

model, preprocess = clip.load("ViT-B/32")
model.cuda().eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

print(preprocess)
def mmmu_clip_preprocess_collate_fn(batch):
    images = torch.stack([preprocess(sample['image_1'].convert('RGB')) for sample in batch])
    texts = torch.stack([clip.tokenize(sample['keywords'])[0] for sample in batch])
    return {'image_1': images, 'keywords': texts}
dataloader = torch.utils.data.DataLoader(dataset_single_image, 
                                         batch_size=64,
                                         collate_fn=mmmu_clip_preprocess_collate_fn)

print(f'num of data: {len(dataloader)}')
# image = Image.open(img_path).convert('RGB')
# image_input = preprocess(image).unsqueeze(0).cuda()
# with torch.no_grad():
#     image_features = model.encode_image(image_input).float()

clip_features = []
print("here")
with torch.no_grad():
    for batch in tqdm(dataloader):
        images = batch['image_1'].to('cuda')
        texts = batch['keywords'].to('cuda')
        # text_inputs = clip.tokenize(batch['conversation'], truncate=True).to('cuda')
        image_features = model.encode_image(images).float()
        text_features = model.encode_text(texts).float()
        # text_features = model.encode_text(text_inputs).float()
        #logits_per_image, logits_per_text = model(images.to('cuda'), text_inputs)
        #probs = logits_per_image.softmax(dim=-1).cpu().numpy()
        #print(image_features)
        #print(probs)
        #text_features = model.encode_text(labels)
        #logits_per_image, logits_per_text = model(image, text)
        #probs = logits_per_image.softmax(dim=-1).cpu().numpy()
        #print(probs)
        #clip_features.append[[image_features,text_features]]
        clip_features.append((image_features.cpu()+text_features.cpu())/2)
print(len(clip_features))
print(clip_features[0])
#filename = 'mean_features.pkl'
#with open(filename, 'wb') as file:
#    pickle.dump(mean_features, file)

In [62]:
clip_features_concat = torch.cat(clip_features, dim=0)
clip_embeddings_file = data_path / f'clip/mmmu_{dir_name}_image_text_clip_embd_cold_start.pkl'
with open(clip_embeddings_file, 'wb') as f:
    pickle.dump(clip_features_concat, f)

In [28]:
val_clip_embeddings_file = '/home/jizej/Workspaces/cache-of-thoughts/data/mmmu/val/clip/cold_start_single_image.pkl'
with open(val_clip_embeddings_file, 'rb') as f:
    val_clip_features = pickle.load(f)

In [ ]:
val_clip_features = np.array(val_clip_features)
val_clip_features = val_clip_features.squeeze(2)
print(val_clip_features.shape)
val_clip_features = torch.tensor(val_clip_features)

In [31]:
val_clip_image_features = val_clip_features[:, 0, :]

In [ ]:
val_clip_image_features.shape

In [37]:
dump_to = '/home/jizej/Workspaces/cache-of-thoughts/data/mmmu/val/clip/mmmu_val_image_clip_embd_cold_start.pkl'
with open(dump_to, 'wb') as f:
    pickle.dump(val_clip_image_features, f)

In [34]:
# read from dump to
with open(dump_to, 'rb') as f:
    clip_features = pickle.load(f)

In [ ]:
clip_features.shape